In [1]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
from sklearn.model_selection import train_test_split

df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "bwandowando/philippine-spam-sms-messages",
    "SPAM_SMS.csv",
)

ham_df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "uciml/sms-spam-collection-dataset",
    "spam.csv",
    pandas_kwargs={"encoding": "latin-1"},
)
ham_only = ham_df[ham_df['v1'] == 'ham'][['v2']].rename(columns={'v2': 'text'})
ham_only['label'] = 'ham'

spam_only = df[['text']].copy()
spam_only['label'] = 'spam'

combined = pd.concat([ham_only, spam_only], ignore_index=True).dropna()
combined = combined[combined['text'].str.strip() != '']

print(combined['label'].value_counts())

X_train, X_test, y_train, y_test = train_test_split(
    combined['text'], combined['label'],
    test_size=0.2, random_state=42, stratify=combined['label']
)
print("Split done:", X_train.shape, X_test.shape)

label
ham     4825
spam    1023
Name: count, dtype: int64
Split done: (4678,) (1170,)


In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', max_features=5000)),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000))
])

pipeline.fit(X_train, y_train)

preds = pipeline.predict(X_test)
print(classification_report(y_test, preds))
print(confusion_matrix(y_test, preds))

              precision    recall  f1-score   support

         ham       0.99      1.00      1.00       965
        spam       1.00      0.98      0.99       205

    accuracy                           0.99      1170
   macro avg       0.99      0.99      0.99      1170
weighted avg       0.99      0.99      0.99      1170

[[964   1]
 [  5 200]]


In [3]:
import joblib
joblib.dump(pipeline, 'spam_classifier.joblib')
print("Model saved as spam_classifier.joblib")

Model saved as spam_classifier.joblib
